In [ ]:
# === Setup ===
# Runtime: <2m with OAI_FAST_MODE=1
# Hardware: CPU smoke; GPU recommended for full run
# Network: optional
# Competition-safe: No — learning profile
import os, random, math, re, json, csv, time
from pathlib import Path
import numpy as np
FAST_MODE = os.getenv("OAI_FAST_MODE", "0") == "1"
RUNTIME_PROFILE = os.getenv("OAI_RUNTIME_PROFILE", "cpu")
random.seed(42)
np.random.seed(42)
print(f"Runtime profile: {'fast' if FAST_MODE else 'full'}")

# IoU, NMS và AP bằng NumPy

Box dùng quy ước liên tục `[x1,y1,x2,y2]`; width=`x2-x1`, không cộng 1.

In [ ]:
def iou(a,b):
    a=np.asarray(a,float); b=np.asarray(b,float)
    wh=np.maximum(0,np.minimum(a[2:],b[2:])-np.maximum(a[:2],b[:2])); inter=wh.prod()
    area_a=np.maximum(0,a[2:]-a[:2]).prod(); area_b=np.maximum(0,b[2:]-b[:2]).prod()
    union=area_a+area_b-inter
    return 0.0 if union<=0 else inter/union

def nms(boxes,scores,threshold=.5):
    order=np.argsort(-np.asarray(scores),kind="stable"); keep=[]
    while len(order):
        current=order[0]; keep.append(int(current))
        order=np.array([j for j in order[1:] if iou(boxes[current],boxes[j])<=threshold],dtype=int)
    return keep

def average_precision(recalls,precisions):
    r=np.r_[0.,recalls,1.]; p=np.r_[0.,precisions,0.]
    p=np.maximum.accumulate(p[::-1])[::-1]; changed=np.where(r[1:]!=r[:-1])[0]
    return np.sum((r[changed+1]-r[changed])*p[changed+1])

assert abs(iou([0,0,10,10],[5,5,15,15])-1/7)<1e-12
boxes=np.array([[0,0,10,10],[1,1,9,9],[20,20,30,30]],float); scores=np.array([.9,.8,.7])
assert nms(boxes,scores,.5)==[0,2]
ap=average_precision(np.array([.5,1.]),np.array([1.,.5])); assert 0<=ap<=1
print("IoU/NMS/AP",iou(boxes[0],boxes[1]),nms(boxes,scores),ap)